In [ ]:
!pip install spacy transformers datasets seqeval
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=893af607976a711965ac1bafc6b61b18bee72450288645bac61814eea62eb117
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 124.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
import json

with open("/content/Kufale.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of parallel sentences:", len(data))
print(data[0])

Number of parallel sentences: 1360
{'gez': 'ዝንቱ ነገረ ኩፋሌ መዋዕላተ ሕግ ወለስምዕ ለግብረ ዓመታት ለተሳብዖቶሙ ለኢዮቤልውሳቲሆሙ ውስተ ኵሉ ዓመታተ ዓለም በከመ ተናገሮ ለሙሴ በደብረ ሲና አመ ዐርገ ይንሣእ ጽላተ እብን ሕግ ወትእዛዝ በቃለ እግዚአብሔር በከመ ይቤሎ ይዕርግ ውስተ ርእሰ ደብር።', 'eng': 'These are the words regarding the divisions of the times of the law and of the testimony, of the events of the years, of the weeks of their jubilees throughout all the years of eternity as he related (them) to Moses on Mt. Sinai when he went up to receive the stone tablets — the law and the commandments — on the Lord’s orders as he had told him that he should come up to the summit of the mountain.'}


In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

allowed_labels = ["PERSON", "LOC", "GPE", "ORG", "MISC"]

def extract_entities(sentence):
    doc = nlp(sentence)
    entities = []
    for ent in doc.ents:
        if ent.label_ in allowed_labels:
            entities.append((ent.text, ent.label_))
    return entities

In [8]:
print(extract_entities(data[0]["eng"]))

[('Mt. Sinai', 'LOC')]


In [9]:
def normalize_geez(token):
    prefixes = ["ወ", "ለ", "በ", "ኀበ"]
    for p in prefixes:
        if token.startswith(p):
            return token[len(p):]
    return token

# BIO Projection

In [10]:
def project_to_geez(eng_sentence, gez_sentence):
    entities = extract_entities(eng_sentence)

    gez_tokens = gez_sentence.strip().split()
    labels = ["O"] * len(gez_tokens)

    for ent_text, ent_label in entities:
        for i, token in enumerate(gez_tokens):
            norm_token = normalize_geez(token)
            if ent_text.lower() in norm_token.lower():
                bio_label = "B-" + map_label(ent_label)
                labels[i] = bio_label

    return gez_tokens, labels


def map_label(label):
    if label == "PERSON":
        return "PER"
    if label in ["LOC", "GPE"]:
        return "LOC"
    if label == "ORG":
        return "ORG"
    return "MISC"

# **Silver CoNLL File**

In [11]:
output_path = "/content/gez_silver.conll"

with open(output_path, "w", encoding="utf-8") as out:
    for item in data:
        gez = item["gez"]
        eng = item["eng"]

        tokens, labels = project_to_geez(eng, gez)

        for t, l in zip(tokens, labels):
            out.write(f"{t} {l}\n")
        out.write("\n")

print("Projection complete. File saved at:", output_path)

Projection complete. File saved at: /content/gez_silver.conll


In [12]:
!head -n 40 /content/gez_silver.conll

ዝንቱ O
ነገረ O
ኩፋሌ O
መዋዕላተ O
ሕግ O
ወለስምዕ O
ለግብረ O
ዓመታት O
ለተሳብዖቶሙ O
ለኢዮቤልውሳቲሆሙ O
ውስተ O
ኵሉ O
ዓመታተ O
ዓለም O
በከመ O
ተናገሮ O
ለሙሴ O
በደብረ O
ሲና O
አመ O
ዐርገ O
ይንሣእ O
ጽላተ O
እብን O
ሕግ O
ወትእዛዝ O
በቃለ O
እግዚአብሔር O
በከመ O
ይቤሎ O
ይዕርግ O
ውስተ O
ርእሰ O
ደብር። O

ወኮነ O
በቀዳሚ O
ዓመት O
በፀአቶሙ O
ለደቂቀ O


In [13]:
from collections import Counter

counter = Counter()

with open("/content/gez_silver.conll", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            label = line.strip().split()[-1]
            counter[label] += 1

print(counter)

Counter({'O': 25820})


In [14]:
all_entities = []

for item in data:
    eng = item.get("eng", "")
    if isinstance(eng, str):
        ents = extract_entities(eng)
        for e in ents:
            all_entities.append(e[0])

from collections import Counter
print(Counter(all_entities).most_common(20))

[('Jacob', 233), ('Abraham', 119), ('Isaac', 112), ('Egypt', 90), ('Joseph', 88), ('Israel', 67), ('Noah', 49), ('Canaan', 49), ('Rebecca', 46), ('Rachel', 32), ('Shem', 31), ('Judah', 31), ('Adam', 30), ('the Garden of Eden', 29), ('Sarah', 28), ('Hebron', 27), ('Laban', 23), ('Bethel', 22), ('Leah', 22), ('Jerusalem', 20)]


# **Co-Occurrence Statistics**

In [15]:
from collections import defaultdict, Counter

entity_geez_counter = defaultdict(Counter)

for item in data:
    eng = item.get("eng", "")
    gez = item.get("gez", "")

    if not isinstance(eng, str) or not isinstance(gez, str):
        continue

    entities = extract_entities(eng)
    gez_tokens = gez.strip().split()

    for ent_text, _ in entities:
        for token in gez_tokens:
            entity_geez_counter[ent_text][token] += 1

In [16]:
entity_mapping = {}

for ent, counter in entity_geez_counter.items():
    most_common = counter.most_common(3)  # top 3 candidates
    entity_mapping[ent] = [w for w, c in most_common]

for k in list(entity_mapping.keys())[:10]:
    print(k, "->", entity_mapping[k])

Mt. Sinai -> ['ውስተ', 'ሲና', 'በደብረ']
Egypt -> ['ግብጽ', 'ውስተ', 'ምድረ']
Abraham -> ['አብርሃም', 'እስመ', 'ከመ']
Isaac -> ['ይስሐቅ', 'ውስተ', 'እስመ']
Jacob -> ['ያዕቆብ', 'ለያዕቆብ', 'እስመ']
Belial -> ['ውሉዶሙ', 'መንፈሰ', 'ከመ']
Israel -> ['እስራኤል', 'እስመ', 'ከመ']
Mt. Zion -> ['ጽዮን', 'እግዚአብሔር', 'ኵሉ']
Jerusalem -> ['እምቅድመ', 'ቅድመ', 'ይሑሩ']
Israelite -> ['እስራኤል', 'ወከመ', 'ፍጥረተ']


# **Filter Noisy Tokens**

In [18]:
# Common Geʽez stopwords to remove
stopwords = {"ወ", "ውስተ", "እስመ", "ከመ", "ለ", "በ", "ኵሉ" ,"እም","ዘ","እንበለ","ውስተ","አምጣነ"}

clean_entity_mapping = {}

for ent, tokens in entity_mapping.items():
    filtered = []
    for token in tokens:
        if token not in stopwords and len(token) > 3:
            filtered.append(token)
    if filtered:
        clean_entity_mapping[ent] = filtered[:2]

# Inspect sample
for k in list(clean_entity_mapping.keys())[:15]:
    print(k, "->", clean_entity_mapping[k])

Mt. Sinai -> ['በደብረ']
Abraham -> ['አብርሃም']
Isaac -> ['ይስሐቅ']
Jacob -> ['ያዕቆብ', 'ለያዕቆብ']
Belial -> ['ውሉዶሙ', 'መንፈሰ']
Israel -> ['እስራኤል']
Mt. Zion -> ['እግዚአብሔር']
Jerusalem -> ['እምቅድመ']
Israelite -> ['እስራኤል', 'ፍጥረተ']
Eden -> ['እምቅድመ']
Israelites -> ['እስራኤል']
fig -> ['ወከደነት', 'ኀፍረታ']
Eve -> ['ብእሲቱ']
Cain -> ['ኢዮቤልዉ']
Abel -> ['ወአቤል', 'እምቅድመ']


# **BIO Projection Using Clean Mapping**

In [19]:
def project_to_geez_with_lexicon(eng_sentence, gez_sentence):
    entities = extract_entities(eng_sentence)

    gez_tokens = gez_sentence.strip().split()
    labels = ["O"] * len(gez_tokens)

    for ent_text, ent_label in entities:
        if ent_text in clean_entity_mapping:
            possible_forms = clean_entity_mapping[ent_text]

            for i, token in enumerate(gez_tokens):
                for form in possible_forms:
                    if form in token:
                        labels[i] = "B-" + map_label(ent_label)

    return gez_tokens, labels

In [20]:
output_path = "/content/gez_silver.conll"

with open(output_path, "w", encoding="utf-8") as out:
    for item in data:
        eng = item.get("eng", "")
        gez = item.get("gez", "")

        if not isinstance(eng, str) or not isinstance(gez, str):
            continue

        if eng.strip() == "" or gez.strip() == "":
            continue

        tokens, labels = project_to_geez_with_lexicon(eng, gez)

        for t, l in zip(tokens, labels):
            out.write(f"{t} {l}\n")
        out.write("\n")

print("Projection complete.")

Projection complete.


In [21]:
from collections import Counter

counter = Counter()

with open("/content/gez_silver.conll", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            label = line.strip().split()[-1]
            counter[label] += 1

print(counter)

Counter({'O': 24623, 'B-PER': 654, 'B-LOC': 380, 'B-ORG': 163})


In [22]:
def project_to_geez_final(eng_sentence, gez_sentence):
    entities = extract_entities(eng_sentence)

    gez_tokens = gez_sentence.strip().split()
    labels = ["O"] * len(gez_tokens)

    for ent_text, ent_label in entities:
        if ent_text in clean_entity_mapping:
            possible_forms = clean_entity_mapping[ent_text]
            for i, token in enumerate(gez_tokens):
                for form in possible_forms:
                    if form in token:
                        labels[i] = "B-" + map_label(ent_label)

    return gez_tokens, labels

In [24]:
output_path = "/content/gez_silver.conll"

with open(output_path, "w", encoding="utf-8") as out:
    for item in data:
        eng = item.get("eng", "")
        gez = item.get("gez", "")

        if not isinstance(eng, str) or not isinstance(gez, str):
            continue

        if eng.strip() == "" or gez.strip() == "":
            continue

        tokens, labels = project_to_geez_final(eng, gez)

        for t, l in zip(tokens, labels):
            out.write(f"{t} {l}\n")
        out.write("\n")

print(" Silver dataset generated:", output_path)

 Silver dataset generated: /content/gez_silver.conll


In [25]:
from collections import Counter

counter = Counter()
with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            label = line.strip().split()[-1]
            counter[label] += 1

print(counter)

Counter({'O': 24623, 'B-PER': 654, 'B-LOC': 380, 'B-ORG': 163})


# **HuggingFace Dataset + Split**

In [26]:
!pip install datasets seqeval

In [27]:
from datasets import Dataset, DatasetDict
import random

# Read CoNLL file
sentences = []
labels = []

with open("/content/gez_silver.conll", "r", encoding="utf-8") as f:
    tokens = []
    token_labels = []
    for line in f:
        line = line.strip()
        if not line:
            if tokens:
                sentences.append(tokens)
                labels.append(token_labels)
                tokens = []
                token_labels = []
        else:
            splits = line.split()
            if len(splits) >= 2:
                token, label = splits[0], splits[1]
                tokens.append(token)
                token_labels.append(label)
    # add last sentence if missing
    if tokens:
        sentences.append(tokens)
        labels.append(token_labels)

print(f"Total sentences: {len(sentences)}")

Total sentences: 1360


In [28]:
unique_labels = set(l for seq in labels for l in seq)
label2id = {l: i for i, l in enumerate(sorted(unique_labels))}
id2label = {i: l for l, i in label2id.items()}

print("Labels:", label2id)

Labels: {'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'O': 3}


In [29]:
encoded_labels = [[label2id[l] for l in seq] for seq in labels]

In [30]:
random.seed(42)
combined = list(zip(sentences, encoded_labels))
random.shuffle(combined)

# Hold 150 sentences for evaluation
eval_size = 150
eval_data = combined[:eval_size]
train_data = combined[eval_size:]

train_sentences, train_labels = zip(*train_data)
eval_sentences, eval_labels = zip(*eval_data)

In [31]:
train_dataset = Dataset.from_dict({"tokens": train_sentences, "labels": train_labels})
eval_dataset = Dataset.from_dict({"tokens": eval_sentences, "labels": eval_labels})

dataset_dict = DatasetDict({
    "train": train_dataset,
    "eval": eval_dataset
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1210
    })
    eval: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 150
    })
})


In [32]:
!pip install transformers

In [41]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.optim import AdamW
from datasets import DatasetDict
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [42]:
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
)
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=4, bias=True)
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
 

In [43]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # For subword tokens inside the same word, use same label
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = torch.tensor(labels)
    return tokenized_inputs

In [45]:
# Install dependencies if not already
!pip install transformers datasets seqeval

In [46]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [47]:
model_name = "xlm-roberta-base"

# Use your label mappings
label2id = {"B-PER":0, "B-LOC":1, "B-ORG":2, "B-MISC":3, "O":4}
id2label = {v:k for k,v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=5, bias=True)
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
 

In [48]:
def tokenize_and_align_labels(sentences, labels, max_len=128):
    input_ids = []
    attention_masks = []
    label_ids = []

    for sent, lab in zip(sentences, labels):
        tokenized = tokenizer(
            sent,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors="pt"
        )
        word_ids = tokenized.word_ids(batch_index=0)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(lab[word_idx])
            else:
                aligned_labels.append(lab[word_idx])
            previous_word_idx = word_idx

        input_ids.append(tokenized["input_ids"][0])
        attention_masks.append(tokenized["attention_mask"][0])
        label_ids.append(torch.tensor(aligned_labels))

    return TensorDataset(torch.stack(input_ids),
                         torch.stack(attention_masks),
                         torch.stack(label_ids))

In [49]:
batch_size = 8

# Assuming dataset_dict already exists from Module 2
train_dataset = tokenize_and_align_labels(dataset_dict["train"]["tokens"],
                                          dataset_dict["train"]["labels"])
eval_dataset  = tokenize_and_align_labels(dataset_dict["eval"]["tokens"],
                                          dataset_dict["eval"]["labels"])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader  = DataLoader(eval_dataset, batch_size=batch_size)

In [50]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [51]:
from tqdm import tqdm

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_loader):.4f}")

Training Epoch 1: 100%|██████████| 152/152 [00:41<00:00,  3.65it/s]


Epoch 1 average loss: 0.2401


Training Epoch 2: 100%|██████████| 152/152 [00:40<00:00,  3.75it/s]


Epoch 2 average loss: 0.1510


Training Epoch 3: 100%|██████████| 152/152 [00:40<00:00,  3.78it/s]


...

Epoch 8 average loss: 0.0730


Training Epoch 9: 100%|██████████| 152/152 [00:40<00:00,  3.78it/s]


Epoch 9 average loss: 0.0679


Training Epoch 10: 100%|██████████| 152/152 [00:40<00:00,  3.78it/s]

Epoch 10 average loss: 0.0553


In [52]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        for p, l in zip(preds, labels):
            p = p.cpu().numpy()
            l = l.cpu().numpy()
            mask = l != -100
            all_preds.extend(p[mask])
            all_labels.extend(l[mask])

y_true = [id2label[l] for l in all_labels]
y_pred = [id2label[p] for p in all_preds]

print(classification_report(y_true, y_pred, labels=[l for l in label2id if l!="O"]))

Evaluating: 100%|██████████| 19/19 [00:01<00:00, 16.07it/s]

              precision    recall  f1-score   support

       B-PER       0.79      0.64      0.71       157
       B-LOC       0.56      0.15      0.24        60
       B-ORG       0.78      0.82      0.80       328
      B-MISC       0.99      0.99      0.99      8460

    accuracy                           0.98      9005
   macro avg       0.78      0.65      0.68      9005
weighted avg       0.97      0.98      0.97      9005



In [53]:
save_path = "/content/geez_ner_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f" Model saved at {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model saved at /content/geez_ner_model


In [59]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load trained model
model_path = "/content/geez_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.to(device)
model.eval()

id2label = model.config.id2label

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [60]:
def predict_entities_full_sentence(sentence):
    """
    Input: full Geʽez sentence (string)
    Output: list of tuples (token, predicted_label)
    """
    # Whitespace tokenize internally
    tokens = sentence.strip().split()

    # Tokenize for model
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    # Align predictions to original tokens
    word_ids = encoding.word_ids(batch_index=0)
    result = []
    previous_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            result.append((tokens[word_idx], id2label[preds[idx]]))
        previous_word_idx = word_idx

    return result

In [56]:
def display_entities(preds):
    """
    Prints tokens colored by entity type
    """
    color_map = {
        "B-PER": "\033[91m",   # Red
        "B-LOC": "\033[94m",   # Blue
        "B-ORG": "\033[92m",   # Green
        "B-MISC": "\033[93m",  # Yellow
        "O": "\033[0m"         # Default
    }
    reset = "\033[0m"
    output = ""
    for token, label in preds:
        output += f"{color_map[label]}{token}{reset} "
    print(output.strip())

In [62]:
sentence = "አብርሃም ሖረ ኃበ ዮርዳኖስ"
preds = predict_entities_full_sentence(sentence)
display_entities(preds)

አብርሃም ሖረ ኃበ ዮርዳኖስ
